<a href="https://colab.research.google.com/github/MawiyaManzar/AI-Engineering/blob/main/MedicalRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# we will use Pinecone for vector DB and integrated embeddings for easier execution.
'''
1. Langchainvectorstore.

'''

In [ ]:
!pip install langchain-text-splitters

In [ ]:
!pip install langchain-pinecone

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
!pip install pinecone datasets -q

In [ ]:
from pinecone import Pinecone
from datasets import load_dataset
from google.colab import userdata

In [ ]:
pc = Pinecone(
    api_key=userdata.get("PINECONE_API_KEY")
)

In [ ]:
index = pc.Index("medical-index")

In [ ]:
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",'en',
    split="train[:100]"
)

In [ ]:
print(dataset)
print(dataset[0])

In [ ]:
print(dataset[0]["Complex_CoT"])

In [ ]:
from pprint import pprint

In [ ]:
records = []

for i, item in enumerate(dataset):

    text = f"""
    Question:
    {item['Question']}

    Reasoning:
    {item['Complex_CoT']}

    Answer:
    {item['Response']}
    """

    records.append({
        "_id":str(i),
        "text":text,
        "category":"medical"
    })

pprint(records[0]["text"][:1000])

In [ ]:
batch_size = 20

for i in range(0, len(records), batch_size):

    batch = records[i:i + batch_size]

    index.upsert_records(
        namespace="medical-rag",
        records=batch
    )

    print(f"Uploaded {i + len(batch)} records")

In [ ]:
results = index.search(
    namespace="medical-rag",
    query={
        "top_k": 3,
        "inputs": {
            "text": "treatment of hypothyroidism in ischemic heart disease"
        }
    }
)

In [ ]:
for hit in results["result"]["hits"]:

    print("SCORE:", hit["_score"])
    print(hit["fields"]["text"][:1000])

    print("=" * 80)

In [ ]:
!pip install openai -q

In [ ]:
from openai import OpenAI
from google.colab import userdata

In [ ]:
client = OpenAI(
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
context = "\n\n".join([
    hit["fields"]["text"]
    for hit in results["result"]["hits"]
])

query = "What is the recommended approach for initiating treatment of hypothyroidism in ischemic heart disease?"

prompt = f"""
You are a medical assistant.

Answer ONLY using the retrieved context.

If the answer is not found in the context, say:
"I could not find the answer in the retrieved context."

Retrieved Context:
{context}

Question:
{query}
"""

In [ ]:
response = client.chat.completions.create(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2
)

In [ ]:
print(response.choices[0].message.content)